# Tutorial 9: ML-Driven Scaling and Model Emulation

## What You Will Learn

- Use the ResourceAdvisor for deterministic recommendations
- Train and use LearnedAdvisor for ML-backed predictions
- Mark functions as emulatable with `@emulatable`
- Dispatch between emulators and full models based on confidence
- Use active learning to improve emulator accuracy

## Prerequisites

- Completed Tutorials 1, 3, and 6
- `pip install scalable[ml]`

> **Note:** Some ML features require historical telemetry. This notebook generates synthetic data where needed.

In [ ]:
import os
import tempfile
import time

project_dir = tempfile.mkdtemp(prefix="scalable-ml-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## Part A: ML-Driven Resource Advising

### Step 1: Deterministic ResourceAdvisor

The baseline advisor uses quantile statistics from telemetry history.

In [ ]:
from scalable import ResourceAdvisor

# The ResourceAdvisor needs run history.
# Let's first generate some telemetry by running a workflow.

from scalable import ScalableSession

manifest = """\
version: 1
project:
  name: ml-demo
targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none
components:
  gcam:
    cpus: 2
    memory: 4G
tasks:
  run_gcam:
    component: gcam
    cache: true
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest)


def simulate_gcam(scenario: int) -> dict:
    time.sleep(0.1 + (scenario % 5) * 0.05)
    return {"scenario": scenario, "demand_mw": scenario * 1.5}


# Generate multiple runs for history
for run_num in range(3):
    sess = ScalableSession.from_yaml("./scalable.yaml", target="local")
    cl = sess.start()
    futs = [cl.submit(simulate_gcam, i, tag="gcam") for i in range(5 + run_num * 2)]
    cl.gather(futs)
    sess.close()
    time.sleep(0.2)

print(f"Generated 3 runs of telemetry data")

In [ ]:
# Now use the ResourceAdvisor
try:
    advisor = ResourceAdvisor.from_history("./.scalable/runs")
    recommendation = advisor.recommend(
        task="run_gcam",
        target="local",
        confidence=0.95,
    )
    
    print(f"ResourceAdvisor Recommendation:")
    print(f"  Task: {recommendation.task}")
    print(f"  Target: {recommendation.target}")
    print(f"  Confidence: {recommendation.confidence}")
    print(f"  Workers: {recommendation.workers}")
    print(f"  Resources: {recommendation.resources}")
    print(f"  Evidence: {recommendation.evidence}")
except Exception as e:
    print(f"Advisor needs more data: {e}")
    print("(This is expected with very few runs)")

### Step 2: LearnedAdvisor (ML-Enhanced)

The `LearnedAdvisor` trains a model on telemetry to predict resources based on task features.

In [ ]:
try:
    from scalable import LearnedAdvisor
    
    advisor = LearnedAdvisor.from_history(
        "./.scalable/runs",
        model_type="random_forest",
    )
    
    recommendation = advisor.recommend(
        task="run_gcam",
        target="local",
    )
    
    print(f"LearnedAdvisor Recommendation:")
    print(f"  Workers: {recommendation.workers}")
    print(f"  Resources: {recommendation.resources}")
    print(f"  Confidence: {recommendation.confidence:.2f}")
    
except ImportError:
    print("LearnedAdvisor requires scalable[ml]")
    print("Install with: pip install scalable[ml]")
except Exception as e:
    print(f"LearnedAdvisor: {e}")
    print("(Needs sufficient history for training)")

### Step 3: Model Types Comparison

| Model | Accuracy | Speed | History Needed |
|-------|----------|-------|---------------|
| `linear` | Low | Fast (<1s) | 5+ runs |
| `random_forest` | Medium | Moderate | 10+ runs |
| `gradient_boosting` | High | Slow (30-120s) | 50+ runs |

### Step 4: AdaptiveScaler with ML

In [ ]:
try:
    from scalable import AdaptiveScaler
    
    scaler = AdaptiveScaler(
        min_workers={"gcam": 1},
        max_workers={"gcam": 20},
        scale_up_threshold=0.7,
        scale_down_threshold=0.3,
        cooldown_seconds=30,
    )
    
    # Simulate evaluation
    decision = scaler.evaluate(
        pending_tasks=[{"tag": "gcam"} for _ in range(15)],
        active_workers={"gcam": 3},
    )
    
    print("AdaptiveScaler Decision:")
    print(f"  Has changes: {decision.has_changes}")
    print(f"  Add workers: {decision.workers_to_add}")
    print(f"  Remove workers: {decision.workers_to_remove}")
    print(f"  Reasoning: {decision.reasoning}")
    print(f"  Confidence: {decision.confidence:.2f}")
    
except ImportError:
    print("AdaptiveScaler requires scalable[ml]")

## Part B: Model Emulation

### Step 5: The @emulatable Decorator

Mark expensive functions as candidates for surrogate model replacement.

In [ ]:
try:
    from scalable.emulation import emulatable
    from scalable.emulation.decorator import _EMULATABLE_REGISTRY, EmulationSpec
    
    @emulatable(
        tag="gcam",
        inputs=["fuel_cost", "population"],
        outputs=["demand_mw"],
        uncertainty="required",
        fallback="full_model",
        domain={
            "fuel_cost": (0, 500),
            "population": (7e9, 12e9),
        },
        confidence_threshold=0.9,
    )
    def run_energy_scenario(fuel_cost, population):
        """Full GCAM scenario — expensive computation."""
        time.sleep(0.5)  # Simulate expensive work
        demand_mw = fuel_cost * 0.07 + population * 3e-9
        return {"demand_mw": demand_mw}
    
    print("Function marked as @emulatable")
    print(f"\nRegistered emulatable functions: {list(_EMULATABLE_REGISTRY.keys())}")
    
    spec = _EMULATABLE_REGISTRY["run_energy_scenario"]
    print(f"\nEmulation spec:")
    print(f"  Tag: {spec.tag}")
    print(f"  Inputs: {spec.inputs}")
    print(f"  Outputs: {spec.outputs}")
    print(f"  Domain: {spec.domain}")
    print(f"  Confidence threshold: {spec.confidence_threshold}")
    print(f"  Fallback: {spec.fallback}")
    
except ImportError:
    print("Emulation requires scalable[ml]")
    print("Install with: pip install scalable[ml]")

### Step 6: Running the Full Model

In [ ]:
try:
    # The decorated function still works normally
    start = time.time()
    result = run_energy_scenario(100, 8e9)
    elapsed = time.time() - start
    
    print(f"Full model result: {result}")
    print(f"Time: {elapsed:.3f}s")
    print("\nWhen an emulator is trained and registered, calls can be")
    print("routed to the fast surrogate instead of the full model.")
except NameError:
    print("(Requires scalable[ml] for @emulatable)")

### Step 7: Emulator Registry and Dispatch

In [ ]:
try:
    from scalable.emulation import EmulatorRegistry, EmulatorDispatch
    
    # Create a registry
    os.makedirs(".scalable/emulators", exist_ok=True)
    registry = EmulatorRegistry(".scalable/emulators")
    
    print(f"Emulator Registry: {registry}")
    print(f"Registered emulators: {registry.list()}")
    
    # Create a dispatch instance
    dispatch = EmulatorDispatch(registry, confidence_threshold=0.9)
    print(f"\nEmulatorDispatch configured:")
    print(f"  Confidence threshold: 0.9")
    print(f"  Dispatch logic:")
    print(f"    1. Check if emulator exists for function")
    print(f"    2. Validate inputs are within domain bounds")
    print(f"    3. Get emulator prediction + confidence")
    print(f"    4. If confidence >= 0.9: return emulator result")
    print(f"    5. Else: fall back to full model")
    
except ImportError:
    print("Emulation requires scalable[ml]")

### Step 8: Active Learning Concepts

Active learning strategically selects training points to maximize emulator accuracy.

In [ ]:
try:
    from scalable.emulation import ActiveLearner
    
    print("Active Learning Acquisition Strategies:")
    print("="*50)
    print()
    print("  'uncertainty'")
    print("    Sample where prediction uncertainty is highest.")
    print("    Best for: Expanding emulator coverage uniformly.")
    print()
    print("  'expected_improvement'")
    print("    Sample where model is likely wrong.")
    print("    Best for: Correcting known weaknesses.")
    print()
    print("  'random'")
    print("    Uniform random sampling.")
    print("    Best for: Baseline comparison.")
    print()
    print("Workflow:")
    print("  1. Train initial emulator on small sample")
    print("  2. Use ActiveLearner.suggest() for next batch")
    print("  3. Run full model on suggested points")
    print("  4. Update emulator with new data")
    print("  5. Repeat until accuracy target is met")
    
except ImportError:
    print("ActiveLearner requires scalable[ml]")

### Step 9: Emulation in Production

Massive speedups by routing confident predictions to emulators:

In [ ]:
# Demonstrate the production pattern (conceptual)
production_pattern = '''
from scalable.emulation import EmulatorDispatch, EmulatorRegistry

registry = EmulatorRegistry(".scalable/emulators")
dispatch = EmulatorDispatch(registry, confidence_threshold=0.9)

results = []
emulated = 0
full_model = 0

for cp in range(0, 500, 10):
    for pop in [8e9, 9e9, 10e9]:
        result = dispatch.predict(
            "run_energy_scenario",
            inputs={"fuel_cost": cp, "population": pop},
        )
        
        if result.source == "emulator":
            emulated += 1
        else:
            full_model += 1
        
        results.append(result.values)

print(f"Emulated: {emulated} ({emulated*100/(emulated+full_model):.0f}%)")
print(f"Full model: {full_model}")
print(f"Time saved: ~{emulated * 30} minutes")
'''

print("Production Emulation Pattern:")
print(production_pattern)

### Step 10: Environment Configuration

In [ ]:
print("ML and Emulation Environment Variables:")
print("="*50)
print()
print(f"  SCALABLE_ML=1              Enable ML features")
print(f"  SCALABLE_ML_CACHE_DIR      ML model cache (.scalable/models)")
print(f"  SCALABLE_EMULATION=0       Enable emulation (set to 1)")
print(f"  SCALABLE_EMULATOR_DIR      Emulator registry (.scalable/emulators)")
print(f"  SCALABLE_EMULATION_CONFIDENCE=0.9  Confidence threshold")

## Summary

### Part A: ML Resource Advising
1. `ResourceAdvisor` — deterministic, quantile-based (always available)
2. `LearnedAdvisor` — ML-trained on telemetry (requires scalable[ml])
3. `AdaptiveScaler` — real-time scaling based on queue depth

### Part B: Model Emulation
4. `@emulatable` — marks functions for surrogate replacement
5. `EmulatorRegistry` — stores trained surrogate models
6. `EmulatorDispatch` — confidence-gated routing
7. `ActiveLearner` — strategic training point selection

## Next Steps

- **Tutorial 10**: AI-assisted workflow composition
- **Tutorial 6**: Feed telemetry to the ML advisor
- **Tutorial 5**: Deploy emulation-backed workflows to cloud

In [ ]:
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print("Cleaned up.")